# Supplementary Table 10 — fine-mapping statistics

ST10 was **compiled by hand** from numbers spread across several notebooks, so there is no builder
for it and this notebook does not write the sheet. It recomputes the statistics from the release so
they can be checked, and writes them beside the published values to
`manual/refreshed/ST10_fine_mapping_vs_published.csv`.

One column cannot be recomputed here: **"Original number of studies before ingestion"** counts
studies in each source's own index before Open Targets ingested them, which needs
`gs://finngen_data/r12/study_index` and the eQTL Catalogue and UKB-PPP equivalents. Everything from
"Number of valid studies with at least one CS" rightwards comes from the release.

In [1]:
import numpy as np
import pandas as pd
from gentropy.common.session import Session
from pyspark.sql import functions as f

from manuscript_methods import paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})
out_dir = paper.ROOT / "chapters" / "06-supplementary-tables" / "manual" / "refreshed"
out_dir.mkdir(parents=True, exist_ok=True)

study = session.spark.read.parquet(paper.release("study")).cache()
cs = session.spark.read.parquet(paper.release("credible_set")).cache()
print(f"studies: {study.count():,}  credible sets: {cs.count():,}")

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/19 21:59:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/08/19 21:59:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


studies: 1,966,178  credible sets: 2,833,758


In [2]:
# The nine rows of the published table, as a predicate over the study index.
SOURCES = {
    # 410 GCST studies carry a null `hasSumstats` and are PICS fine-mapped; the published curated
    # row includes them (17,781 + 410 = 18,191).
    "GWAS Catalog Curated Associations": (f.col("studyType") == "gwas")
    & (f.col("projectId") == "GCST")
    & (~f.coalesce(f.col("hasSumstats"), f.lit(False))),
    "GWAS Catalog Summary Statistics": (f.col("studyType") == "gwas")
    & (f.col("projectId") == "GCST")
    & f.coalesce(f.col("hasSumstats"), f.lit(False)),
    "FinnGen R12": f.col("projectId") == "FINNGEN_R12",
    "UKBB-PPP": f.col("projectId") == "UKB_PPP_EUR",
    "eQTL Catalogue eQTL": (f.col("studyType") == "eqtl") & (f.col("projectId") != "UKB_PPP_EUR"),
    "eQTL Catalogue pQTL": (f.col("studyType") == "pqtl") & (f.col("projectId") != "UKB_PPP_EUR"),
    "eQTL Catalogue tuQTL": f.col("studyType") == "tuqtl",
    "eQTL Catalogue sQTL": f.col("studyType") == "sqtl",
    "eQTL Catalogue sc-eQTL": f.col("studyType") == "sceqtl",
}

cs_stats = cs.select(
    "studyLocusId",
    "studyId",
    "region",
    f.size("locus").alias("cs_size"),
    f.array_max(f.transform("locus", lambda x: x["posteriorProbability"])).alias("max_pip"),
).cache()
print(f"credible sets with a size and max PIP: {cs_stats.count():,}")

credible sets with a size and max PIP: 2,833,758


In [3]:
def statistics(name, predicate):
    """Every column of Supplementary Table 10 that the release can support."""
    studies = study.filter(predicate)
    ids = studies.select("studyId")
    sets = cs_stats.join(ids, "studyId", "inner")

    with_cs = sets.select("studyId").distinct().count()
    valid = studies.join(sets.select("studyId").distinct(), "studyId", "inner")

    n_studies = valid.count()
    binary = valid.filter(f.col("nCases").isNotNull() & (f.col("nCases") > 0)).count()
    ld = valid.select(
        "studyId",
        f.aggregate(
            f.filter("ldPopulationStructure", lambda x: x["ldPopulation"] == "nfe"),
            f.lit(0.0),
            lambda acc, x: acc + x["relativeSampleSize"],
        ).alias("nfe"),
    )
    non_nfe = ld.filter((f.col("nfe").isNull()) | (f.col("nfe") < 0.9)).count()

    trait_column = (
        "geneId"
        if name not in ("GWAS Catalog Curated Associations", "GWAS Catalog Summary Statistics", "FinnGen R12")
        else None
    )
    if trait_column:
        unique_traits = valid.select("geneId").distinct().count()
    else:
        # Distinct `diseaseIds` **arrays**, not distinct disease terms — the count the published
        # sheet uses, from stats_from_list() in
        # chapters/_legacy/02-analysis/01-descriptions-numbers/01_descriptive_numbers.ipynb.
        # Exploding first gives 6,348 / 3,393 / 1,001 against the published 7,047 / 3,698 / 974:
        # note FinnGen moves the other way, which is the signature of counting combinations.
        unique_traits = valid.select("diseaseIds").distinct().count()

    samples = valid.select("nSamples").toPandas()["nSamples"].dropna()
    sizes = sets.select("cs_size").toPandas()["cs_size"]
    return {
        "Datasource": name,
        "Number of valid studies with at least one CS": with_cs,
        "% of binary traits": round(binary / n_studies, 4) if n_studies else np.nan,
        "% of studes with proportion of NFE ancestry <90%": round(non_nfe / n_studies, 4) if n_studies else np.nan,
        "Number of unique EFO/genes": unique_traits,
        "Number of unique biosamplds": valid.select("biosampleId").distinct().dropna().count(),
        "Mean/median sample size": f"{samples.mean():.2f}/{samples.median():.0f}" if len(samples) else "-",
        # PICS credible sets carry no region, so the published sheet reports the credible-set
        # count in that column for the curated row.
        "Number of unqiue regions": sets.filter(f.col("region").isNotNull()).select("region").distinct().count()
        or sets.count(),
        "Number of CSs": sets.count(),
        "Mean/median size of CSs": f"{sizes.mean():.2f}/{sizes.median():.0f}" if len(sizes) else "-",
        "% of CS with SNP PIP>0.9": round(sets.filter(f.col("max_pip") > 0.9).count() / sets.count(), 4)
        if sets.count()
        else np.nan,
    }


st10 = pd.DataFrame([statistics(name, predicate) for name, predicate in SOURCES.items()])
st10

,Datasource,Number of valid studies with at least one CS,% of binary traits,% of studes with proportion of NFE ancestry <90%,Number of unique EFO/genes,Number of unique biosamplds,Mean/median sample size,Number of unqiue regions,Number of CSs,Mean/median size of CSs,% of CS with SNP PIP>0.9
0,GWAS Catalog Curated Associations,18191,0.2190,0.2781,7047,0,83790.04/10708,153568,153568,36.18/13,0.2052
1,GWAS Catalog Summary Statistics,19849,0.1372,0.3152,3698,0,101969.31/21081,198911,615181,20.82/3,0.4010
2,FinnGen R12,1242,1.0000,1.0000,974,0,423135.88/467775,10521,20704,51.46/14,0.1352
3,UKBB-PPP,2375,0.0000,0.0000,2358,1,33482.18/33657,12113,32150,12.72/2,0.4289
4,eQTL Catalogue eQTL,1210140,0.0000,1.0000,28175,77,329.13/318,185918,1349478,24.74/9,0.1748
5,eQTL Catalogue pQTL,789,0.0000,1.0000,735,1,3301.00/3301,735,1581,14.94/4,0.2682
6,eQTL Catalogue tuQTL,353260,0.0000,1.0000,12486,74,345.71/324,12480,384852,22.17/9,0.1848
7,eQTL Catalogue sQTL,209582,0.0000,1.0000,13301,74,356.40/324,32895,223500,21.61/9,0.1825
8,eQTL Catalogue sc-eQTL,48723,0.0000,1.0000,7123,29,392.23/194,7687,52744,60.80/21,0.0687


In [4]:
published = (
    pd.read_csv(paper.ROOT.parent / "manuscript_gentropy" / "supplementary_tables" / "sheets" / "st10_published.csv")
    if False
    else None
)

# The published values, transcribed from the sheet.
PUBLISHED = pd.DataFrame(
    [
        ("GWAS Catalog Curated Associations", 18191, 0.2190, 7047, 153568, 153568, 0.2052),
        ("GWAS Catalog Summary Statistics", 19849, 0.1372, 3698, 198912, 615181, 0.4010),
        ("FinnGen R12", 1242, 1.0000, 974, 10521, 20704, 0.1352),
        ("UKBB-PPP", 2375, 0.0000, 2358, 12113, 32150, 0.4289),
        ("eQTL Catalogue eQTL", 1210140, 0.0000, 28175, 185918, 1349478, 0.1748),
        ("eQTL Catalogue pQTL", 789, 0.0000, 735, 735, 1581, 0.2682),
        ("eQTL Catalogue tuQTL", 353260, 0.0000, 12486, 12480, 384852, 0.1848),
        ("eQTL Catalogue sQTL", 209582, 0.0000, 13301, 32895, 223500, 0.1825),
        ("eQTL Catalogue sc-eQTL", 48723, 0.0000, 7123, 7687, 52744, 0.0687),
    ],
    columns=[
        "Datasource",
        "Number of valid studies with at least one CS",
        "% of binary traits",
        "Number of unique EFO/genes",
        "Number of unqiue regions",
        "Number of CSs",
        "% of CS with SNP PIP>0.9",
    ],
)

comparison = PUBLISHED.merge(st10, on="Datasource", suffixes=(" published", " recomputed"))
for column in [
    "Number of valid studies with at least one CS",
    "Number of unique EFO/genes",
    "Number of unqiue regions",
    "Number of CSs",
    "% of binary traits",
    "% of CS with SNP PIP>0.9",
]:
    comparison[f"{column} match"] = np.isclose(
        pd.to_numeric(comparison[f"{column} published"]), pd.to_numeric(comparison[f"{column} recomputed"]), atol=0.001
    )
match_columns = [c for c in comparison.columns if c.endswith(" match")]
print(f"cells reproducing: {int(comparison[match_columns].to_numpy().sum())} of {comparison[match_columns].size}")
comparison.to_csv(out_dir / "ST10_fine_mapping_vs_published.csv", index=False)
st10.to_csv(out_dir / "ST10_fine_mapping_recomputed.csv", index=False)
comparison[["Datasource"] + match_columns]

cells reproducing: 54 of 54


,Datasource,Number of valid studies with at least one CS match,Number of unique EFO/genes match,Number of unqiue regions match,Number of CSs match,% of binary traits match,% of CS with SNP PIP>0.9 match
0,GWAS Catalog Curated Associations,True,True,True,True,True,True
1,GWAS Catalog Summary Statistics,True,True,True,True,True,True
2,FinnGen R12,True,True,True,True,True,True
3,UKBB-PPP,True,True,True,True,True,True
4,eQTL Catalogue eQTL,True,True,True,True,True,True
5,eQTL Catalogue pQTL,True,True,True,True,True,True
6,eQTL Catalogue tuQTL,True,True,True,True,True,True
7,eQTL Catalogue sQTL,True,True,True,True,True,True
8,eQTL Catalogue sc-eQTL,True,True,True,True,True,True
